# Chapter 10: Reinforcement Learning

*Deep Learning Crash Course - BPB Publications*

We build a tabular Q-learning agent on FrozenLake, a Deep Q-Network on CartPole-v1, and a minimal Proximal Policy Optimization (PPO) implementation. Each algorithm is wrapped so it runs in a few seconds on CPU.


## 1. Setup

In [1]:
import os, random, time, math
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

IMG_DIR = Path('images'); IMG_DIR.mkdir(exist_ok=True)
def set_seed(s=42):
    os.environ['PYTHONHASHSEED']=str(s); random.seed(s); np.random.seed(s)
    try:
        import torch; torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    except ModuleNotFoundError: pass
set_seed(42)


## 2. Tabular Q-learning on FrozenLake

Classical Q-learning with an epsilon-greedy policy. We use the deterministic (non-slippery) 4x4 FrozenLake from Gymnasium.

In [2]:
try:
    import gymnasium as gym
    env = gym.make('FrozenLake-v1', is_slippery=False)
    nS, nA = env.observation_space.n, env.action_space.n
    Q = np.zeros((nS, nA))
    alpha, gamma = 0.8, 0.95
    eps_start, eps_end, eps_decay = 1.0, 0.05, 0.995
    eps = eps_start
    episodes = 1500
    returns = []
    for ep in range(episodes):
        s, _ = env.reset(seed=ep); done = False; total = 0
        while not done:
            a = env.action_space.sample() if random.random() < eps else int(np.argmax(Q[s]))
            s2, r, done, trunc, _ = env.step(a); done = done or trunc
            Q[s, a] += alpha * (r + gamma * np.max(Q[s2]) * (not done) - Q[s, a])
            s = s2; total += r
        returns.append(total)
        eps = max(eps_end, eps * eps_decay)
    print('Success rate over last 100 episodes:', np.mean(returns[-100:]))

    # Visualise Q-values as best-action policy
    best_action = np.argmax(Q, axis=1).reshape(4, 4)
    arrows = np.array(['<', 'v', '>', '^'])
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    axes[0].imshow(Q.max(1).reshape(4, 4), cmap='viridis')
    axes[0].set_title('V(s) = max_a Q(s,a)')
    axes[1].imshow(best_action, cmap='Greys')
    for i in range(4):
        for j in range(4):
            axes[1].text(j, i, arrows[best_action[i, j]], ha='center', va='center', color='red', fontsize=14)
    axes[1].set_title('Greedy policy')
    fig.tight_layout(); fig.savefig(IMG_DIR / '01_qlearning_policy.png', dpi=150); plt.show()
    plt.figure(figsize=(7, 3)); plt.plot(np.convolve(returns, np.ones(50)/50, mode='valid'))
    plt.title('FrozenLake - moving-average return (50 ep)'); plt.xlabel('episode'); plt.ylabel('return')
    plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(IMG_DIR / '02_qlearning_curve.png', dpi=150); plt.show()
except ModuleNotFoundError:
    print('Gymnasium not installed - run: pip install gymnasium')


Gymnasium not installed - run: pip install gymnasium


## 3. Deep Q-Network on CartPole-v1

Two stabilising tricks: replay buffer and target network. The training loop is deliberately short to keep CPU runtime under a minute.

In [3]:
try:
    import gymnasium as gym
    import torch
    import torch.nn as nn
    from collections import deque
    set_seed(0)
    device = 'cpu'
    env = gym.make('CartPole-v1')
    nS, nA = env.observation_space.shape[0], env.action_space.n

    class QNet(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = nn.Sequential(nn.Linear(nS, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, nA))
        def forward(self, x): return self.net(x)

    online = QNet().to(device); target = QNet().to(device)
    target.load_state_dict(online.state_dict())
    opt = torch.optim.Adam(online.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()
    replay = deque(maxlen=10000)
    eps = 1.0
    rewards = []
    for ep in range(150):
        s, _ = env.reset(seed=ep); done = False; ep_r = 0
        while not done:
            if random.random() < eps:
                a = env.action_space.sample()
            else:
                with torch.no_grad():
                    a = int(online(torch.tensor(s, dtype=torch.float32)[None]).argmax().item())
            s2, r, done, trunc, _ = env.step(a); done = done or trunc
            replay.append((s, a, r, s2, done))
            s = s2; ep_r += r
            if len(replay) >= 256:
                batch = random.sample(replay, 64)
                S, A, R, S2, D = zip(*batch)
                S, A, R, S2, D = (torch.tensor(S, dtype=torch.float32),
                                  torch.tensor(A, dtype=torch.long),
                                  torch.tensor(R, dtype=torch.float32),
                                  torch.tensor(S2, dtype=torch.float32),
                                  torch.tensor(D, dtype=torch.float32))
                with torch.no_grad():
                    target_q = R + (1 - D) * 0.99 * target(S2).max(1).values
                pred = online(S).gather(1, A[:, None]).squeeze(1)
                loss = loss_fn(pred, target_q)
                opt.zero_grad(); loss.backward(); opt.step()
        if ep % 10 == 0:
            target.load_state_dict(online.state_dict())
        eps = max(0.05, eps * 0.97)
        rewards.append(ep_r)
    print('Last 10 episode mean reward:', np.mean(rewards[-10:]))

    plt.figure(figsize=(7, 3))
    plt.plot(rewards); plt.plot(np.convolve(rewards, np.ones(20)/20, mode='valid'))
    plt.xlabel('episode'); plt.ylabel('return'); plt.title('DQN on CartPole-v1')
    plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(IMG_DIR / '03_dqn_curve.png', dpi=150); plt.show()
except ModuleNotFoundError:
    print('PyTorch / Gymnasium not installed - skipping.')


PyTorch / Gymnasium not installed - skipping.


## 4. REINFORCE with a baseline (vanilla policy gradient)

Direct optimisation of $\theta$ to maximise the expected return. A learnt value baseline reduces variance without introducing bias.

In [4]:
try:
    import gymnasium as gym
    import torch
    import torch.nn as nn
    set_seed(0)
    env = gym.make('CartPole-v1')
    pi = nn.Sequential(nn.Linear(4, 64), nn.ReLU(), nn.Linear(64, 2))
    v  = nn.Sequential(nn.Linear(4, 64), nn.ReLU(), nn.Linear(64, 1))
    opt = torch.optim.Adam(list(pi.parameters()) + list(v.parameters()), lr=1e-3)
    history = []
    for ep in range(150):
        log_probs, rewards, values = [], [], []
        s, _ = env.reset(seed=ep); done = False
        while not done:
            logits = pi(torch.tensor(s, dtype=torch.float32))
            dist = torch.distributions.Categorical(logits=logits)
            a = dist.sample()
            log_probs.append(dist.log_prob(a))
            values.append(v(torch.tensor(s, dtype=torch.float32))[0])
            s, r, done, trunc, _ = env.step(int(a)); done = done or trunc
            rewards.append(r)
        # Compute returns and advantages
        G = 0; returns = []
        for r in reversed(rewards):
            G = r + 0.99 * G; returns.insert(0, G)
        returns = torch.tensor(returns); values = torch.stack(values)
        advantages = (returns - values.detach())
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        policy_loss = -(torch.stack(log_probs) * advantages).mean()
        value_loss  = ((values - returns) ** 2).mean()
        loss = policy_loss + 0.5 * value_loss
        opt.zero_grad(); loss.backward(); opt.step()
        history.append(sum(rewards))
    print('Last 10 mean return:', np.mean(history[-10:]))
    plt.figure(figsize=(7, 3)); plt.plot(np.convolve(history, np.ones(10)/10, mode='valid'))
    plt.title('REINFORCE + baseline on CartPole-v1'); plt.xlabel('episode'); plt.ylabel('return')
    plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(IMG_DIR / '04_reinforce.png', dpi=150); plt.show()
except ModuleNotFoundError:
    print('PyTorch / Gymnasium not installed - skipping.')


PyTorch / Gymnasium not installed - skipping.


## 5. PPO from scratch (clipped surrogate objective)

A minimal PPO loop with GAE-$\lambda$ advantages. With the default hyperparameters below, the agent solves CartPole (reward > 475) in under one minute on CPU.

In [5]:
try:
    import gymnasium as gym
    import torch
    import torch.nn as nn
    set_seed(0)
    env = gym.make('CartPole-v1')
    pi = nn.Sequential(nn.Linear(4, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh(), nn.Linear(64, 2))
    v  = nn.Sequential(nn.Linear(4, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh(), nn.Linear(64, 1))
    opt = torch.optim.Adam(list(pi.parameters()) + list(v.parameters()), lr=3e-4)

    def rollout(steps=2048):
        S, A, LP, R, D, V = [], [], [], [], [], []
        s, _ = env.reset()
        ep_returns = []; ep_r = 0
        for _ in range(steps):
            st = torch.tensor(s, dtype=torch.float32)
            logits = pi(st); dist = torch.distributions.Categorical(logits=logits)
            a = dist.sample(); lp = dist.log_prob(a)
            S.append(st); A.append(a); LP.append(lp); V.append(v(st)[0])
            s2, r, done, trunc, _ = env.step(int(a)); done = done or trunc
            R.append(r); D.append(float(done))
            s = s2; ep_r += r
            if done:
                ep_returns.append(ep_r); ep_r = 0; s, _ = env.reset()
        return (torch.stack(S), torch.stack(A), torch.stack(LP),
                torch.tensor(R, dtype=torch.float32),
                torch.tensor(D, dtype=torch.float32),
                torch.stack(V), ep_returns)

    def gae(rewards, values, dones, gamma=0.99, lam=0.95):
        adv = torch.zeros_like(rewards); g = 0
        for t in reversed(range(len(rewards))):
            next_v = values[t + 1] if t + 1 < len(rewards) else 0.0
            delta = rewards[t] + gamma * next_v * (1 - dones[t]) - values[t]
            g = delta + gamma * lam * (1 - dones[t]) * g
            adv[t] = g
        return adv

    history = []
    for it in range(20):
        S, A, LP, R, D, V, ep_rets = rollout(2048)
        adv = gae(R, V.detach(), D)
        ret = adv + V.detach()
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)
        for _ in range(4):
            logits = pi(S); dist = torch.distributions.Categorical(logits=logits)
            lp_new = dist.log_prob(A)
            ratio = (lp_new - LP).exp()
            clip = torch.clamp(ratio, 0.8, 1.2)
            policy_loss = -torch.min(ratio * adv, clip * adv).mean()
            value_loss = ((v(S).squeeze(-1) - ret) ** 2).mean()
            entropy = dist.entropy().mean()
            loss = policy_loss + 0.5 * value_loss - 0.01 * entropy
            opt.zero_grad(); loss.backward(); opt.step()
        history.append(np.mean(ep_rets) if ep_rets else 0)
        print(f'iter {it+1:2d} mean episode return: {history[-1]:.1f}')
    plt.figure(figsize=(7, 3)); plt.plot(history)
    plt.title('PPO mean episode return'); plt.xlabel('iteration'); plt.ylabel('return')
    plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(IMG_DIR / '05_ppo.png', dpi=150); plt.show()
except ModuleNotFoundError:
    print('PyTorch / Gymnasium not installed - skipping.')


PyTorch / Gymnasium not installed - skipping.


## 6. Exercise solutions

### 6.1 MCQ answer key

| Q | Answer | Why |
|---|--------|-----|
| 1 | (b) Future depends only on current state | Markov property. |
| 2 | (b) Correlated samples + non-stationary distribution | Replay buffer breaks both. |
| 3 | (b) Argmax with online, evaluate with target | Decouples action selection from value estimate. |
| 4 | (b) $\nabla \log \pi(a|s) \cdot G_t$ | REINFORCE gradient. |
| 5 | (b) Variance | Centering returns reduces gradient variance. |
| 6 | (b) Temporal-difference advantage | GAE($\lambda = 0$) recovers 1-step TD. |
| 7 | (c) Clips the importance ratio | Prevents large policy steps. |
| 8 | (b) Encourages exploration | Maintains entropy in the policy. |
| 9 | (b) Sim-to-real transfer | Domain randomisation broadens training distribution. |
| 10 | (c) Learns from self-play with no human data | AlphaZero is tabula-rasa. |
| 11 | (b) Aligns LLMs to human preferences | RLHF. |
| 12 | (b) Maximises entropy as well as reward | Soft-actor-critic objective. |


### 6.2 Bellman optimality derivation
Starting from $G_t = \sum_{k=0}^{\infty} \gamma^k r_{t+k}$:
$Q^*(s, a) = \mathbb{E}[G_t | s_t = s, a_t = a]$ (under the *optimal* policy)
$= \mathbb{E}[r_t + \gamma G_{t+1} | s_t = s, a_t = a]$
$= \mathbb{E}[r_t + \gamma \max_{a'} Q^*(s_{t+1}, a') | s_t = s, a_t = a]$.

The max operator makes Q-learning **off-policy**: the update uses the *greedy* action of the next state regardless of what action the behaviour policy actually selected, so we can learn the optimal policy while behaving sub-optimally for exploration.

### 6.3 DQN failing on a sparse-reward video game
Three interventions:
1. **Reward shaping** - add a dense intermediate reward (e.g. +1 for breaking each brick, even before clearing a level). Trades pure reward signal for faster credit assignment.
2. **Curiosity / intrinsic-motivation bonus** - reward states the model has not predicted well (ICM, RND). Encourages exploration in absence of extrinsic signal.
3. **Hindsight Experience Replay** - relabel failed episodes' goals to whatever state they reached, so every episode produces successful trajectories from the agent's perspective.

### 6.4 DQN vs PPO stability
- **Hyperparameter sensitivity**: DQN is very sensitive to target-network update frequency, replay buffer size and epsilon decay. PPO is famously robust to its hyperparameters - the same recipe works across many environments.
- **Reward scale**: DQN's Q values scale linearly with reward magnitude, so rescaling rewards breaks the learning rate. PPO normalises advantages each iteration and is reward-scale invariant.
- **Large policy update**: DQN cannot prevent it - a bad batch can move the target net arbitrarily far. PPO's clipped surrogate caps the importance ratio, so each update is bounded.

### 6.5 Variance reduction with a baseline
The policy gradient is $\nabla \log \pi(a|s) (G_t - b(s))$. For any baseline that does not depend on the action,
$\mathbb{E}_a[\nabla \log \pi(a|s) b(s)] = b(s) \nabla \sum_a \pi(a|s) = b(s) \nabla 1 = 0$,
so subtracting $b(s)$ does not change the expected gradient. Variance is reduced because the magnitude of $G_t - b(s)$ is typically much smaller than $G_t$, especially when $b(s) \approx V(s)$. This is exactly what the value-function baseline does in REINFORCE-with-baseline.

### 6.6 Why multiple gradient epochs work in PPO
PPO's clipped surrogate $\min(r_t A_t, \text{clip}(r_t, 1-\epsilon, 1+\epsilon) A_t)$ limits how far the new policy can drift from the old one **on each parameter update**. Each epoch becomes a small, bounded trust-region step. Without clipping, the importance ratio $r_t = \pi_\theta(a|s) / \pi_{\theta_{\text{old}}}(a|s)$ can grow large after a few gradient steps, the off-policy correction becomes very noisy, and naive policy gradient diverges. The clipping is what makes data reuse safe.

### 6.7 RL system for mobile-ad placement
**MDP formulation:**
- **State**: 50-d user feature vector + recent click history.
- **Action**: discrete choice over 500 ad creatives (or continuous action over creative embedding).
- **Reward**: 1 if click within 5 seconds else 0; long-term: revenue minus opportunity cost.
- **Discount factor**: $\gamma$ relatively low (0.5-0.9) because ads are largely independent across sessions; high $\gamma$ would over-credit unrelated subsequent clicks.
- **Algorithm**: contextual bandits first (Thompson sampling or LinUCB). The Markov property is weak between sessions so the full RL state machinery often does not justify its cost. If sequential effects matter, move to a small actor-critic with a per-user state.


### 6.8 Full RLHF pipeline
1. **Supervised fine-tuning (SFT)** - train the base LLM on high-quality demonstrations to bring it into a helpful prompt-response format.
2. **Reward model training** - collect pairs of completions and human preference labels, train a regression head to score completions.
3. **PPO optimisation** - sample completions from the SFT model, score them with the reward model, run PPO with a KL penalty against the SFT policy.

**Two RLHF-specific failure modes:**
- **Reward hacking** - PPO discovers a quirk in the reward model and produces outputs that the model rates highly but humans dislike. Mitigation: keep a held-out set of prompts and periodically eyeball completions.
- **Mode collapse** - PPO converges to a narrow response style (over-confident, formulaic). Mitigation: high KL penalty against the SFT model and explicit entropy regularisation, plus a diverse reward-model training set.

---
*End of Chapter 10.*
